# CICIDS2017: Decision Tree (baseline and class-weighted)

Runs two configurations and compares them at the end:

1. **Baseline**: default settings, no class weighting.
2. **Class-weighted**: identical, except that errors on rare classes count more (`balanced` weights: each class's weight is inversely proportional to how many training rows it has).

Entropy criterion, as described in the report.

**Before running:** run `CICIDS2017_Preprocessing.ipynb` once. It creates `cicids2017_grouped.npz`, which this notebook loads. Keep that file in the same folder as this notebook (or change `DATA_FILE`), then run all cells.

Expect a few minutes per run: the training set has about 2 million rows. It needs roughly 2 GB of RAM.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_fscore_support, f1_score)

SEED = 0
DATASET_NAME = "CICIDS2017"

# The file written by CICIDS2017_Preprocessing.ipynb (keep it in the same folder as this notebook)
DATA_FILE = "cicids2017_grouped.npz"

## Load the preprocessed data

In [ ]:
d = np.load(DATA_FILE)
X_train, y_train = d["X_train"], d["y_train"]
X_test, y_test = d["X_test"], d["y_test"]
class_names = list(d["class_names"])
n_classes = len(class_names)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(dict(zip(class_names, np.bincount(y_train, minlength=n_classes).tolist())))

In [ ]:
def evaluate(model_name, config, y_true, y_pred, class_names):
    """Print the metrics, classification report and confusion matrix; return the numbers."""
    labels = np.arange(len(class_names))
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)

    print(f"{model_name} ({DATASET_NAME}, {config})")
    print(f"Accuracy {acc:.4f} | Macro precision {macro_p:.3f} | Macro recall {macro_r:.3f} | "
          f"Macro F1 {macro_f1:.3f} | Weighted F1 {weighted_f1:.3f}\n")
    print(classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                zero_division=0))

    fig, ax = plt.subplots(figsize=(9, 7))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=labels),
                           display_labels=class_names).plot(ax=ax, xticks_rotation=45,
                                                           cmap="Blues", colorbar=False)
    ax.set_title(f"{model_name} confusion matrix ({DATASET_NAME}, {config})")
    fig.tight_layout()
    plt.show()

    return {"accuracy": acc, "macro_precision": macro_p, "macro_recall": macro_r,
            "macro_f1": macro_f1, "weighted_f1": weighted_f1,
            "report": classification_report(y_true, y_pred, labels=labels,
                                            target_names=class_names, zero_division=0,
                                            output_dict=True)}

## Model

In [ ]:
MODEL_NAME = "Decision Tree"

def run(config, class_weight):
    model = DecisionTreeClassifier(criterion="entropy", random_state=SEED,
                                   class_weight=class_weight)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = time.perf_counter() - start
    print(f"Trained in {train_seconds:.1f}s | depth {model.get_depth()} | leaves {model.get_n_leaves()}\n")

    result = evaluate(MODEL_NAME, config, y_test, model.predict(X_test), class_names)
    result["train_seconds"] = train_seconds
    return result

results = {}

## 1. Baseline

In [ ]:
results["baseline"] = run("baseline", class_weight=None)

## 2. Class-weighted

In [ ]:
class_weights = compute_class_weight("balanced", classes=np.arange(n_classes), y=y_train)
print("Class weights used:")
print(pd.Series(class_weights, index=class_names).round(2).to_string())

results["class_weighted"] = run("class-weighted", class_weight="balanced")

## 3. Comparison

In [ ]:
rows = {"baseline": "Baseline", "class_weighted": "Class-weighted"}

summary = pd.DataFrame({
    rows[k]: {"Accuracy": r["accuracy"], "Macro precision": r["macro_precision"],
              "Macro recall": r["macro_recall"], "Macro F1": r["macro_f1"],
              "Weighted F1": r["weighted_f1"], "Train time (s)": r["train_seconds"]}
    for k, r in results.items()}).T
display(summary.round(4))

b, w = results["baseline"]["report"], results["class_weighted"]["report"]
per_class = pd.DataFrame({
    "Precision (baseline)": [b[c]["precision"] for c in class_names],
    "Precision (weighted)": [w[c]["precision"] for c in class_names],
    "Recall (baseline)": [b[c]["recall"] for c in class_names],
    "Recall (weighted)": [w[c]["recall"] for c in class_names],
    "F1 (baseline)": [b[c]["f1-score"] for c in class_names],
    "F1 (weighted)": [w[c]["f1-score"] for c in class_names],
}, index=class_names)
display(per_class.round(2))

ax = per_class[["F1 (baseline)", "F1 (weighted)"]].plot.bar(figsize=(10, 4), rot=45)
ax.set_ylabel("F1-score")
ax.set_title(f"{MODEL_NAME}: per-class F1, baseline vs class-weighted ({DATASET_NAME})")
plt.tight_layout()
plt.show()